## **MoE v11 — 7-Expert Pipeline**

### Arsenal completo:

| Expert | Modelo | Papel |
|--------|--------|-------|
| GPT-OSS 120B | gpt-oss-120b | LLM mais forte, 120B params |
| Llama 70B | llama-3.3-70b-versatile | LLM forte, 70B params |
| Kimi | kimi-k2-instruct-0905 | LLM forte, neutro |
| Qwen | qwen3-32b (no_think) | LLM bom, thinking desligado |
| Scout | llama-4-scout-17b | LLM fraco, baixo peso |
| Stats v2 | Local (regras) | 50+ features, zero API |
| Profile | Local (aprendido) | Perfis do support set, zero API |

**Pass 1 (Human vs AI):** 4 LLMs (sem Scout) → voto ponderado

**Pass 2 (Qual IA):** 5 LLMs + 2 locais = **7 experts** → gating adaptativo

### **1. Imports**

In [13]:
import pandas as pd
import numpy as np
import time
import json
import re
import math
import string
from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from groq import Groq

sns.set_style('whitegrid')
LABELS = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
AI_LABELS = ['Anthropic', 'Google', 'Meta', 'OpenAI']
print('Imports OK')

Imports OK


### **2. Keys + Modelos**

In [14]:
GROQ_KEYS = [
    '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', 
    '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',
    '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',
    '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',
]

groq_clients = [Groq(api_key=key) for key in GROQ_KEYS if key]
groq_idx = 0

GPT_OSS_MODEL = 'openai/gpt-oss-120b'
LLAMA70_MODEL = 'llama-3.3-70b-versatile'
KIMI_MODEL    = 'moonshotai/kimi-k2-instruct-0905'
QWEN_MODEL    = 'qwen/qwen3-32b'
SCOUT_MODEL   = 'meta-llama/llama-4-scout-17b-16e-instruct'

print(f'Keys: {len(groq_clients)}')
print(f'Models: GPT-OSS-120B, Llama-3.3-70B, Kimi-K2, Qwen3-32B, Scout-17B')

Keys: 4
Models: GPT-OSS-120B, Llama-3.3-70B, Kimi-K2, Qwen3-32B, Scout-17B


### **3. Dados**

In [15]:
df_support = pd.read_csv('../database/dataset-subm1-labels.csv', sep=';')
df_support.columns = df_support.columns.str.strip().str.lower()

df_test = pd.read_csv('../database/dataset-samples.csv', sep=';')
df_test.columns = df_test.columns.str.strip().str.lower()

N_PER_CLASS = 8
support_set = pd.concat([
    group.sample(min(N_PER_CLASS, len(group)), random_state=42)
    for _, group in df_support.groupby('label')
]).reset_index(drop=True)

support_ai = support_set[support_set['label'] != 'Human'].reset_index(drop=True)

print(f'Few-shot total: {len(support_set)} | Teste: {len(df_test)}')
print(f'Few-shot AI-only: {len(support_ai)}')
if 'label' in df_test.columns:
    print(f'Distribuicao teste: {dict(df_test["label"].value_counts())}')

Few-shot total: 40 | Teste: 125
Few-shot AI-only: 32
Distribuicao teste: {'Human': np.int64(52), 'Anthropic': np.int64(23), 'Meta': np.int64(17), 'OpenAI': np.int64(17), 'Google': np.int64(16)}


### **4. Feature Extraction (50+ features)**

In [16]:
def extract_features(text):
    feat = {}
    words = text.split()
    n_words = len(words)
    n_chars = len(text)
    lines = text.split('\n')
    n_lines = len(lines)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s for s in sentences if len(s.strip()) > 3]
    n_sentences = max(len(sentences), 1)
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    n_paragraphs = max(len(paragraphs), 1)
    
    feat['n_words'] = n_words
    feat['n_chars'] = n_chars
    feat['n_sentences'] = n_sentences
    feat['n_paragraphs'] = n_paragraphs
    feat['n_lines'] = n_lines
    feat['avg_word_len'] = np.mean([len(w) for w in words]) if words else 0
    feat['avg_sentence_len'] = n_words / n_sentences
    feat['std_sentence_len'] = np.std([len(s.split()) for s in sentences]) if len(sentences) > 1 else 0
    feat['avg_paragraph_len'] = n_words / n_paragraphs
    
    words_lower = [w.lower().strip(string.punctuation) for w in words]
    words_lower = [w for w in words_lower if w]
    unique_words = set(words_lower)
    feat['ttr'] = len(unique_words) / max(len(words_lower), 1)
    feat['hapax_ratio'] = sum(1 for w, c in Counter(words_lower).items() if c == 1) / max(len(words_lower), 1)
    feat['long_word_ratio'] = sum(1 for w in words_lower if len(w) > 10) / max(len(words_lower), 1)
    
    feat['bullet_lines'] = len(re.findall(r'^\s*[-*\u2022]\s', text, re.MULTILINE))
    feat['numbered_lines'] = len(re.findall(r'^\s*\d+[.)]\s', text, re.MULTILINE))
    feat['list_ratio'] = (feat['bullet_lines'] + feat['numbered_lines']) / max(n_lines, 1)
    feat['bold_count'] = len(re.findall(r'\*\*[^*]+\*\*', text))
    feat['header_count'] = len(re.findall(r'^#{1,6}\s', text, re.MULTILINE))
    feat['has_code_block'] = int('```' in text)
    feat['formatting_density'] = (feat['bold_count'] + feat['header_count'] + feat['bullet_lines'] + feat['numbered_lines']) / max(n_lines, 1)
    
    feat['em_dashes'] = text.count('\u2014') + text.count(' -- ') + text.count('\u2013')
    feat['semicolons'] = text.count(';')
    feat['exclamations'] = text.count('!')
    feat['questions'] = text.count('?')
    feat['parentheses'] = text.count('(') + text.count(')')
    feat['commas_per_sentence'] = text.count(',') / n_sentences
    
    openai_p = [r'\bcertainly\b', r'\babsolutely\b', r'\bmoreover\b', r'\bfurthermore\b',
        r'\bgreat question\b', r'\bin addition\b', r'\bcomprehensive\b',
        r'\blet me\b', r'\bhere.s a breakdown\b', r'\bin conclusion\b',
        r'\bto summarize\b', r'\bdelve\b', r'\btapestry\b',
        r'\bparadigm\b', r'\bfacet\b', r'\blandscape\b', r'\brobust\b',
        r'\bseamless\b', r'\bleverage\b', r'\bpivotal\b', r'\bfostering\b',
        r'\bintricate\b', r'\bunderscores\b', r'\bcommendable\b',
        r'\bmultifaceted\b', r'\bevergreen\b']
    feat['openai_phrases'] = sum(len(re.findall(p, text, re.IGNORECASE)) for p in openai_p)
    
    anthropic_p = [r'\bworth noting\b', r'\barguably\b', r'\bit.s important to consider\b',
        r'\bthat said\b', r'\bnuanced\b', r'\bon the other hand\b',
        r'\bto be fair\b', r'\bit.s worth\b', r'\bkeep in mind\b',
        r'\bhowever\b', r'\bI should\b', r'\bI.d suggest\b',
        r'\bI think\b', r'\bperspective\b', r'\bthoughtful\b',
        r'\breasonable\b', r'\bone could argue\b', r'\bI.d recommend\b']
    feat['anthropic_phrases'] = sum(len(re.findall(p, text, re.IGNORECASE)) for p in anthropic_p)
    
    google_p = [r'\bhere is\b', r'\bhere are\b', r'\bhere.s\b',
        r'\bkey (point|aspect|factor|takeaway)s?\b', r'\bin summary\b', r'\bspecifically\b']
    feat['google_phrases'] = sum(len(re.findall(p, text, re.IGNORECASE)) for p in google_p)
    
    meta_p = [r'\bin today.s (world|digital|fast|modern)\b', r'\bit.s no secret\b',
        r'\bthe fact (is|that)\b', r'\bwhen it comes to\b', r'\bat the end of the day\b',
        r'\bthe (truth|reality|bottom line) is\b', r'\bin (this|the) (article|post|guide|blog)\b']
    feat['meta_phrases'] = sum(len(re.findall(p, text, re.IGNORECASE)) for p in meta_p)
    
    if len(words_lower) > 4:
        bigrams = [' '.join(words_lower[i:i+2]) for i in range(len(words_lower)-1)]
        trigrams = [' '.join(words_lower[i:i+3]) for i in range(len(words_lower)-2)]
        feat['repeated_bigrams'] = sum(1 for c in Counter(bigrams).values() if c > 1) / max(len(bigrams), 1)
        feat['repeated_trigrams'] = sum(1 for c in Counter(trigrams).values() if c > 1) / max(len(trigrams), 1)
    else:
        feat['repeated_bigrams'] = 0
        feat['repeated_trigrams'] = 0
    
    feat['has_url'] = int(bool(re.search(r'https?://', text)))
    feat['has_citation'] = int(bool(re.search(r'\[\d+\]|\(\d{4}\)|et al\.', text)))
    feat['first_person'] = len(re.findall(r'\b(I|my|me|mine|myself)\b', text)) / max(n_words, 1)
    feat['second_person'] = len(re.findall(r'\b(you|your|yours|yourself)\b', text, re.IGNORECASE)) / max(n_words, 1)
    
    confident = len(re.findall(r'\b(clearly|obviously|definitely|undoubtedly|certainly|absolutely|always|never)\b', text, re.IGNORECASE))
    hedging = len(re.findall(r'\b(perhaps|maybe|possibly|might|could|somewhat|arguably|likely|tends? to|generally|often|sometimes|typically)\b', text, re.IGNORECASE))
    feat['confidence_ratio'] = confident / max(confident + hedging, 1)
    feat['hedging_density'] = hedging / n_sentences
    
    transitions = len(re.findall(r'\b(however|moreover|furthermore|additionally|consequently|therefore|thus|nevertheless|nonetheless|meanwhile|subsequently|accordingly|hence)\b', text, re.IGNORECASE))
    feat['transition_density'] = transitions / n_sentences
    
    if n_chars > 0:
        char_freq = Counter(text.lower())
        char_probs = [c / n_chars for c in char_freq.values()]
        feat['char_entropy'] = -sum(p * math.log2(p) for p in char_probs if p > 0)
    else:
        feat['char_entropy'] = 0
    
    if sentences:
        starters = [s.strip().split()[0].lower() if s.strip().split() else '' for s in sentences]
        feat['sentence_starter_diversity'] = len(set(starters)) / max(len(starters), 1)
    else:
        feat['sentence_starter_diversity'] = 0
    
    return feat

print(f'Features: {len(extract_features("Test sentence here."))}')

Features: 40


### **5. Profile Expert (aprende do support set)**

In [17]:
class ProfileExpert:
    def __init__(self):
        self.profiles = {}
        self.feat_names = []
        self.feat_weights = {}
    
    def fit(self, texts, labels):
        feat_by_label = defaultdict(list)
        all_feats = []
        for text, label in zip(texts, labels):
            f = extract_features(text)
            feat_by_label[label].append(f)
            all_feats.append(f)
        self.feat_names = list(all_feats[0].keys())
        for label, feats_list in feat_by_label.items():
            profile = {}
            for fn in self.feat_names:
                vals = [f[fn] for f in feats_list]
                profile[fn] = (np.mean(vals), max(np.std(vals), 0.01))
            self.profiles[label] = profile
        for fn in self.feat_names:
            means = [self.profiles[l][fn][0] for l in self.profiles]
            stds = [self.profiles[l][fn][1] for l in self.profiles]
            self.feat_weights[fn] = (max(means) - min(means)) / max(np.mean(stds), 0.001)
        max_w = max(self.feat_weights.values()) if self.feat_weights else 1
        self.feat_weights = {k: v / max_w for k, v in self.feat_weights.items()}
        top = sorted(self.feat_weights.items(), key=lambda x: x[1], reverse=True)[:10]
        print('Top 10 features:')
        for fn, w in top:
            vals = ' | '.join(f'{l}={self.profiles[l][fn][0]:.2f}' for l in sorted(self.profiles.keys()))
            print(f'  {fn:30s} w={w:.2f} | {vals}')
    
    def predict(self, text, valid_labels=None):
        if valid_labels is None:
            valid_labels = list(self.profiles.keys())
        feats = extract_features(text)
        scores = {}
        for label in valid_labels:
            if label not in self.profiles:
                scores[label] = -999; continue
            score = 0
            for fn in self.feat_names:
                mean, std = self.profiles[label][fn]
                z = abs(feats.get(fn, 0) - mean) / std
                score -= z * self.feat_weights.get(fn, 0.5)
            scores[label] = score
        winner = max(scores, key=scores.get)
        return winner, scores

print('=== Profile Expert (AI only) ===')
profile_expert = ProfileExpert()
profile_expert.fit(support_ai['text'].tolist(), support_ai['label'].tolist())

print('\n=== Profile Expert (5 classes) ===')
profile_expert_full = ProfileExpert()
profile_expert_full.fit(support_set['text'].tolist(), support_set['label'].tolist())

=== Profile Expert (AI only) ===
Top 10 features:
  char_entropy                   w=1.00 | Anthropic=4.27 | Google=4.25 | Meta=4.16 | OpenAI=4.26
  ttr                            w=0.90 | Anthropic=0.81 | Google=0.80 | Meta=0.70 | OpenAI=0.78
  hapax_ratio                    w=0.76 | Anthropic=0.70 | Google=0.69 | Meta=0.55 | OpenAI=0.66
  avg_word_len                   w=0.69 | Anthropic=6.32 | Google=6.06 | Meta=5.51 | OpenAI=6.29
  repeated_bigrams               w=0.63 | Anthropic=0.02 | Google=0.00 | Meta=0.04 | OpenAI=0.02
  n_chars                        w=0.62 | Anthropic=779.38 | Google=765.62 | Meta=624.62 | OpenAI=779.75
  anthropic_phrases              w=0.59 | Anthropic=0.38 | Google=0.00 | Meta=0.00 | OpenAI=0.12
  em_dashes                      w=0.56 | Anthropic=0.75 | Google=0.12 | Meta=0.00 | OpenAI=0.12
  std_sentence_len               w=0.54 | Anthropic=6.38 | Google=6.03 | Meta=8.52 | OpenAI=4.32
  transition_density             w=0.52 | Anthropic=0.05 | Google=0.0

### **6. Expert Estatístico v2**

In [18]:
def statistical_expert_v2(text):
    scores = {l: 0.0 for l in AI_LABELS}
    feats = extract_features(text)
    if feats['n_words'] < 5: return None, scores
    
    # OPENAI
    if feats['openai_phrases'] >= 3: scores['OpenAI'] += 4.0
    elif feats['openai_phrases'] >= 2: scores['OpenAI'] += 2.5
    elif feats['openai_phrases'] >= 1: scores['OpenAI'] += 1.0
    if feats['formatting_density'] > 0.2: scores['OpenAI'] += 2.0
    elif feats['formatting_density'] > 0.1: scores['OpenAI'] += 1.0
    if feats['bold_count'] >= 3: scores['OpenAI'] += 2.0
    elif feats['bold_count'] >= 1: scores['OpenAI'] += 1.0
    if feats['list_ratio'] > 0.2: scores['OpenAI'] += 2.0
    elif feats['list_ratio'] > 0.1: scores['OpenAI'] += 1.0
    if feats['transition_density'] > 0.4: scores['OpenAI'] += 1.5
    if feats['n_words'] > 400 and feats['formatting_density'] > 0.05: scores['OpenAI'] += 1.0
    # ANTHROPIC
    if feats['anthropic_phrases'] >= 3: scores['Anthropic'] += 4.0
    elif feats['anthropic_phrases'] >= 2: scores['Anthropic'] += 2.5
    elif feats['anthropic_phrases'] >= 1: scores['Anthropic'] += 1.0
    if feats['em_dashes'] >= 3: scores['Anthropic'] += 3.0
    elif feats['em_dashes'] >= 1: scores['Anthropic'] += 1.5
    if feats['hedging_density'] > 0.5: scores['Anthropic'] += 2.0
    elif feats['hedging_density'] > 0.2: scores['Anthropic'] += 1.0
    if feats['anthropic_phrases'] >= 1 and feats['bold_count'] == 0: scores['Anthropic'] += 1.5
    if feats['parentheses'] >= 4: scores['Anthropic'] += 1.0
    # GOOGLE
    if feats['google_phrases'] >= 2: scores['Google'] += 3.0
    elif feats['google_phrases'] >= 1: scores['Google'] += 2.0
    if feats['n_words'] < 200 and feats['formatting_density'] < 0.05 and feats['bold_count'] == 0:
        scores['Google'] += 2.0
    elif feats['n_words'] < 300 and feats['formatting_density'] < 0.08: scores['Google'] += 1.0
    if feats['ttr'] > 0.6 and feats['n_words'] < 300: scores['Google'] += 1.0
    if feats['hedging_density'] < 0.1 and feats['confidence_ratio'] > 0.5: scores['Google'] += 1.0
    # META
    if feats['repeated_trigrams'] > 0.06: scores['Meta'] += 3.5
    elif feats['repeated_trigrams'] > 0.04: scores['Meta'] += 2.0
    elif feats['repeated_bigrams'] > 0.1: scores['Meta'] += 1.5
    if feats['ttr'] < 0.42: scores['Meta'] += 3.0
    elif feats['ttr'] < 0.48: scores['Meta'] += 1.5
    if feats['meta_phrases'] >= 2: scores['Meta'] += 3.0
    elif feats['meta_phrases'] >= 1: scores['Meta'] += 1.5
    if (feats['openai_phrases'] == 0 and feats['anthropic_phrases'] == 0 
        and feats['bold_count'] == 0 and feats['header_count'] == 0
        and feats['list_ratio'] == 0 and feats['em_dashes'] == 0):
        scores['Meta'] += 1.5
    if feats['sentence_starter_diversity'] < 0.5: scores['Meta'] += 1.5
    
    if max(scores.values()) == 0: return None, scores
    return max(scores, key=scores.get), scores

print('Stats v2 OK')

Stats v2 OK


### **7. Calibração local**

In [19]:
ai_only = support_set[support_set['label'] != 'Human']
print('=== Calibração no Support Set (AI-only) ===')
for name, fn in [('Stats v2', lambda t: statistical_expert_v2(t)[0]),
                  ('Profile',  lambda t: profile_expert.predict(t, AI_LABELS)[0])]:
    hits = sum(1 for _, r in ai_only.iterrows() if fn(r['text']) == r['label'])
    print(f'  {name:12s}: {hits}/{len(ai_only)} = {hits/len(ai_only):.0%}')

=== Calibração no Support Set (AI-only) ===
  Stats v2    : 11/32 = 34%
  Profile     : 27/32 = 84%


### **8. Prompts**

In [20]:
TRUNC_EX = 800
TRUNC_Q  = 1500

SYS_BINARY = """You classify text as Human-written or AI-generated.
Answer with ONLY one word: Human or AI.

HUMAN signals: personal anecdotes, subjective opinions with "I think/feel", typos/grammar errors,
informal tone, inconsistent formatting, URLs/citations, specific personal experiences,
colloquialisms, contractions, emotional language, incomplete thoughts.

AI signals: polished and error-free, comprehensive coverage, structured with headers/bullets,
balanced perspectives, formal transitions (Moreover, Furthermore, However),
no personal experiences, systematic organization, consistent tone throughout,
disclaimers, overly thorough answers."""

def build_binary_prompt(text, support_examples):
    ex = ''
    for _, row in support_examples.iterrows():
        label = 'Human' if row['label'] == 'Human' else 'AI'
        ex += f'Text: {row["text"][:TRUNC_EX]}\nAnswer: {label}\n\n'
    return f'{ex}Text: {text[:TRUNC_Q]}\nAnswer:'

SYS_AI_TYPE = """You are a forensic AI text analyst. Determine which AI wrote the text.

OPENAI (GPT): Signature words: "Certainly!", "Absolutely!", "delve", "tapestry", "landscape", "robust", "seamless", "leverage", "pivotal", "fostering", "multifaceted", "underscores". Heavy formatting: bold (**), numbered lists, bullets. Long, structured. Uses "Moreover", "Furthermore". Ends with summary.

ANTHROPIC (Claude): Em-dashes (\u2014), parenthetical asides. Hedging: "worth noting", "arguably", "that said", "I should mention", "I'd suggest", "to be fair". Balanced, nuanced. Polite but NOT enthusiastic. Less formatting, flowing prose. Uses "I think", "I'd recommend".

GOOGLE (Gemini): Short, direct, factual. "Here is", "Here are" at start. Minimal formatting and filler. Gets to point quickly.

META (LLaMA): Repetitive vocabulary. "In today's world", "It's no secret", "When it comes to". Less polished, simpler structure. No sophisticated formatting.

Your VERY LAST WORD must be exactly one of: Anthropic, Google, Meta, OpenAI"""

def build_ai_type_prompt(text, support_ai_examples):
    ex = ''
    for _, row in support_ai_examples.iterrows():
        ex += f'Text: {row["text"][:TRUNC_EX]}\nAnswer: {row["label"]}\n\n'
    return f'{ex}Text: {text[:TRUNC_Q]}\nAnswer:'

print('Prompts OK')

Prompts OK


### **9. API Wrapper**

In [21]:
def ask_groq(prompt, system_prompt, model, expert_name='Expert', max_tok=150, max_retries=8):
    global groq_idx
    attempt = 0
    
    # Qwen: desligar thinking para evitar rate limits
    if 'qwen' in model.lower():
        prompt = prompt + '\n/no_think'
    
    while attempt < max_retries:
        current_idx = groq_idx % len(groq_clients)
        client = groq_clients[current_idx]
        groq_idx += 1
        try:
            response = client.chat.completions.create(
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': prompt},
                ],
                model=model,
                max_tokens=max_tok,
                temperature=0.0,
            )
            raw = response.choices[0].message.content
            if raw is None: raise ValueError('None response')
            return raw.strip()
        except KeyboardInterrupt: raise
        except Exception as e:
            attempt += 1
            err = str(e)[:120]
            wait = min(20 * attempt, 120)
            print(f'    {expert_name} (K{current_idx+1}): retry {attempt}/{max_retries} wait={wait}s [{type(e).__name__}: {err}]')
            if attempt >= max_retries:
                print(f'    {expert_name}: FAILED')
                return None
            time.sleep(wait)
    return None

def normalize(raw, valid_labels=LABELS):
    if not raw: return None
    clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    clean = re.sub(r'<think>.*$', '', clean, flags=re.DOTALL).strip()
    stripped = re.sub(r'[\*_#:\-\.",\'!\(\)\[\]]', ' ', clean).strip()
    stripped = re.sub(r'\s+', ' ', stripped).strip()
    
    for label in valid_labels:
        if stripped.lower() == label.lower(): return label
    last_words = stripped.split()[-5:] if stripped.split() else []
    for word in reversed(last_words):
        for label in valid_labels:
            if word.lower() == label.lower(): return label
    for label in valid_labels:
        if re.search(rf'\b{label}\b', clean, re.IGNORECASE): return label
    for label in valid_labels:
        if label.lower() in clean.lower(): return label
    if set(valid_labels) == {'Human', 'AI'}:
        for kw in ['artificial', 'generated', 'machine', 'bot', 'llm', 'gpt']:
            if kw in clean.lower(): return 'AI'
        for kw in ['person', 'human-written', 'organic']:
            if kw in clean.lower(): return 'Human'
    return None

print('Wrappers OK')

Wrappers OK


### **10. Teste de Conectividade**

In [22]:
print('A testar modelos...')
test_p = build_binary_prompt('The quick brown fox jumps.', support_set.head(3))

models_to_test = [
    ('GPT-OSS',  GPT_OSS_MODEL, 150),
    ('Llama70B', LLAMA70_MODEL, 100),
    ('Kimi',     KIMI_MODEL,    80),
    ('Qwen',     QWEN_MODEL,    150),
    ('Scout',    SCOUT_MODEL,   80),
]

available_models = {}
for name, model, tok in models_to_test:
    t0 = time.time()
    r = ask_groq(test_p, SYS_BINARY, model, name, max_tok=tok, max_retries=3)
    dt = time.time() - t0
    if r is not None:
        pred = normalize(r, ['Human', 'AI'])
        print(f'  OK   {name:10s}: "{r[:50].replace(chr(10), " ")}" -> {pred} ({dt:.1f}s)')
        available_models[name] = model
    else:
        print(f'  FAIL {name:10s}: indisponivel ({dt:.1f}s)')
    time.sleep(4)

print(f'\nDisponveis: {list(available_models.keys())}')
if len(available_models) < 3:
    print('AVISO: menos de 3 modelos disponíveis!')

A testar modelos...
  OK   GPT-OSS   : "" -> None (0.8s)
  OK   Llama70B  : "Human" -> Human (0.2s)
  OK   Kimi      : "AI" -> AI (0.5s)
  OK   Qwen      : "<think>  </think>  Human" -> Human (0.4s)
  OK   Scout     : "AI" -> AI (0.1s)

Disponveis: ['GPT-OSS', 'Llama70B', 'Kimi', 'Qwen', 'Scout']


### **11. Gating Adaptativo**

In [23]:
PASS1_WEIGHTS = {'gptoss': 1.3, 'llama70': 1.2, 'kimi': 1.2, 'qwen': 1.1, 'scout': 0.7}

def pass1_vote(preds):
    votes = Counter()
    for name, pred in preds.items():
        if pred: votes[pred] += PASS1_WEIGHTS.get(name, 1.0)
    return votes.most_common(1)[0][0] if votes else 'AI'

PASS2_WEIGHTS = {
    'gptoss': 1.4, 'llama70': 1.2, 'kimi': 1.2, 'qwen': 1.1,
    'scout': 0.5, 'stats': 1.0, 'profile': 0.9,
}
SCOUT_BIAS = {'Meta', 'OpenAI'}

def pass2_vote(expert_preds):
    votes = Counter()
    raw = {}
    for name, pred in expert_preds.items():
        if pred is None or pred not in AI_LABELS: continue
        raw[name] = pred
        w = PASS2_WEIGHTS.get(name, 1.0)
        if name == 'scout' and pred in SCOUT_BIAS: w *= 0.15
        votes[pred] += w
    
    if not votes: return 'OpenAI', {'method': 'default'}
    
    counts = Counter(raw.values())
    # Consenso forte (4+)
    for label, c in counts.most_common():
        if c >= 4: return label, {'method': 'strong_consensus', 'count': c}
    # Consenso (3)
    for label, c in counts.most_common():
        if c >= 3: return label, {'method': 'consensus', 'count': c}
    # 2 LLMs fortes + 1 local
    strong = {k: v for k, v in raw.items() if k in ('gptoss','llama70','kimi','qwen')}
    local = {k: v for k, v in raw.items() if k in ('stats','profile')}
    for label, c in Counter(strong.values()).most_common():
        if c >= 2 and label in local.values():
            return label, {'method': 'llm+local', 'count': c}
    # Alta divergência → boost locais
    if len(counts) >= 4:
        bv = Counter()
        for name, pred in raw.items():
            w = PASS2_WEIGHTS.get(name, 1.0)
            if name in ('stats','profile'): w *= 1.5
            if name == 'scout' and pred in SCOUT_BIAS: w *= 0.15
            bv[pred] += w
        return bv.most_common(1)[0][0], {'method': 'divergent_boost', 'votes': dict(bv)}
    
    return votes.most_common(1)[0][0], {'method': 'weighted', 'votes': dict(votes)}

print('Gating OK (7 experts)')

Gating OK (7 experts)


### **12. Pipeline Principal**

In [ ]:
SLEEP = 8

def run_pipeline(df_data, support_df, support_ai_df):
    results = []
    hits_moe = 0
    total = 0

    for idx, row in tqdm(df_data.iterrows(), total=len(df_data), desc='MoE v11'):
        text = row['text']
        true_label = row.get('label', None)
        
        # ═══ PASS 1: Human vs AI ═══
        prompt_bin = build_binary_prompt(text, support_df)
        
        bp = {}  # binary preds
        bp['gptoss'] = normalize(ask_groq(prompt_bin, SYS_BINARY, GPT_OSS_MODEL, 'GPT-OSS', 100), ['Human','AI'])
        bp['llama70'] = normalize(ask_groq(prompt_bin, SYS_BINARY, LLAMA70_MODEL, 'Llama70', 100), ['Human','AI'])
        bp['kimi']    = normalize(ask_groq(prompt_bin, SYS_BINARY, KIMI_MODEL,    'Kimi',    80),  ['Human','AI'])
        bp['qwen']    = normalize(ask_groq(prompt_bin, SYS_BINARY, QWEN_MODEL,    'Qwen',    150), ['Human','AI'])
        
        binary_result = pass1_vote(bp)
        
        ai_preds = {}
        if binary_result == 'Human':
            final_pred = 'Human'
            p2_info = {'method': 'pass1_human'}
        else:
            # ═══ PASS 2: Qual IA (7 experts) ═══
            prompt_ai = build_ai_type_prompt(text, support_ai_df)
            
            ai_preds['gptoss']  = normalize(ask_groq(prompt_ai, SYS_AI_TYPE, GPT_OSS_MODEL, 'GPT-OSS', 200), AI_LABELS)
            ai_preds['llama70'] = normalize(ask_groq(prompt_ai, SYS_AI_TYPE, LLAMA70_MODEL, 'Llama70', 200), AI_LABELS)
            ai_preds['kimi']    = normalize(ask_groq(prompt_ai, SYS_AI_TYPE, KIMI_MODEL,    'Kimi',    250), AI_LABELS)
            ai_preds['qwen']    = normalize(ask_groq(prompt_ai, SYS_AI_TYPE, QWEN_MODEL,    'Qwen',    200), AI_LABELS)
            ai_preds['scout']   = normalize(ask_groq(prompt_ai, SYS_AI_TYPE, SCOUT_MODEL,   'Scout',   200), AI_LABELS)
            
            ai_preds['stats'], _   = statistical_expert_v2(text)
            ai_preds['profile'], _ = profile_expert.predict(text, AI_LABELS)
            
            final_pred, p2_info = pass2_vote(ai_preds)
        
        rec = {
            'id': row.get('id', idx), 'true_label': true_label,
            'bin_gptoss': bp.get('gptoss'), 'bin_llama70': bp.get('llama70'),
            'bin_kimi': bp.get('kimi'), 'bin_qwen': bp.get('qwen'),
            'binary': binary_result, 'moe_pred': final_pred,
            'vote_info': json.dumps(p2_info),
        }
        for k, v in ai_preds.items(): rec[f'ai_{k}'] = v
        results.append(rec)
        
        if true_label:
            total += 1
            if final_pred == true_label: hits_moe += 1
            ok = 'Y' if final_pred == true_label else 'X'
            acc = f'{hits_moe/total:.0%}'
            method = p2_info.get('method', '?')[:12]
            if true_label == 'Human':
                bin_str = ' '.join(f'{k[0].upper()}={v}' for k,v in bp.items() if v)
                print(f'  [{ok}] #{total:3d} {true_label:9s} BIN:{bin_str} -> {final_pred:9s} Acc={acc}')
            else:
                p2_str = ' '.join(f'{k[:2]}={str(v):9s}' for k,v in ai_preds.items() if v)
                print(f'  [{ok}] #{total:3d} {true_label:9s} {p2_str} -> {final_pred:9s} [{method}] Acc={acc}')
        
        time.sleep(SLEEP)

    return pd.DataFrame(results)

print(f'Pipeline v11 pronto | Sleep={SLEEP}s')
print(f'  Pass 1: 4 LLMs | Pass 2: 5 LLMs + 2 locais')

df_results = run_pipeline(df_test, support_set, support_ai)
df_results.to_csv('moe_results_v11.csv', index=False, sep=';')
print(f'\nDone! {len(df_results)} amostras.')

Pipeline v11 pronto | Sleep=8s
  Pass 1: 4 LLMs | Pass 2: 5 LLMs + 2 locais


MoE v11:   0%|          | 0/125 [00:00<?, ?it/s]

  [Y] #  1 Human     BIN:L=Human K=Human Q=Human -> Human     Acc=100%
  [X] #  2 Meta      ll=OpenAI    ki=Google    qw=OpenAI    sc=OpenAI    st=Meta      pr=Meta      -> OpenAI    [consensus] Acc=50%
  [X] #  3 Google    ll=OpenAI    ki=OpenAI    qw=OpenAI    sc=OpenAI    st=Google    pr=OpenAI    -> OpenAI    [strong_conse] Acc=33%
  [X] #  4 Meta      ll=OpenAI    ki=OpenAI    qw=OpenAI    sc=OpenAI    st=Google    pr=Anthropic -> OpenAI    [strong_conse] Acc=25%
  [Y] #  5 Human     BIN:L=Human K=Human Q=Human -> Human     Acc=40%
  [X] #  6 Google     -> Human     [pass1_human] Acc=33%
  [Y] #  7 OpenAI    ll=Meta      ki=OpenAI    qw=OpenAI    sc=OpenAI    st=Google    pr=Anthropic -> OpenAI    [consensus] Acc=43%
  [X] #  8 Meta      ll=OpenAI    ki=Google    qw=OpenAI    sc=OpenAI    st=Google    pr=Anthropic -> OpenAI    [consensus] Acc=38%
  [X] #  9 Google    ll=OpenAI    qw=OpenAI    sc=OpenAI    st=Google    pr=Anthropic -> OpenAI    [consensus] Acc=33%
  [X] # 10 Human 

### **13. Avaliação**

In [ ]:
def evaluate(df_res, pred_col, title=''):
    valid = df_res.dropna(subset=[pred_col, 'true_label'])
    preds, golds = valid[pred_col].tolist(), valid['true_label'].tolist()
    if not preds: return 0.0
    acc = sum(p == g for p, g in zip(preds, golds)) / len(preds)
    print(f'\n{"="*60}\n{title} — Accuracy: {acc:.2%} ({sum(p==g for p,g in zip(preds,golds))}/{len(preds)})\n{"="*60}')
    print(classification_report(golds, preds, labels=LABELS, zero_division=0))
    cm = confusion_matrix(golds, preds, labels=LABELS)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_xlabel('Previsao'); ax.set_ylabel('Real'); ax.set_title(f'{title} — {acc:.2%}')
    plt.tight_layout(); plt.show()
    return acc

acc_moe = evaluate(df_results, 'moe_pred', 'MoE v11 (7 experts)')

if 'true_label' in df_results.columns:
    df_results['true_binary'] = df_results['true_label'].apply(lambda x: 'Human' if x == 'Human' else 'AI')
    bv = df_results.dropna(subset=['binary', 'true_binary'])
    print(f'\nPass 1 (Human vs AI): {(bv["true_binary"]==bv["binary"]).mean():.2%}')

print('\nPor classe:')
for label in LABELS:
    s = df_results[df_results['true_label'] == label]
    if len(s) > 0:
        a = (s['moe_pred'] == label).mean()
        print(f'  {label:10s}: {a:.0%} ({(s["moe_pred"]==label).sum()}/{len(s)})')

ai_rows = df_results[df_results['true_label'] != 'Human'].dropna(subset=['true_label'])
if len(ai_rows) > 0:
    print('\nPor expert (Pass 2):')
    for col in [c for c in df_results.columns if c.startswith('ai_')]:
        v = ai_rows.dropna(subset=[col])
        if len(v) > 0:
            print(f'  {col:15s}: {(v[col]==v["true_label"]).mean():.0%} ({(v[col]==v["true_label"]).sum()}/{len(v)})')

### **14. Análise de Erros**

In [ ]:
if 'true_label' in df_results.columns:
    errors = df_results[df_results['moe_pred'] != df_results['true_label']].copy()
    print(f'Erros: {len(errors)}/{len(df_results)}\n')
    if len(errors) > 0:
        errors['confusion'] = errors['true_label'] + ' -> ' + errors['moe_pred']
        print('Confusoes:')
        for p, c in errors['confusion'].value_counts().head(10).items():
            print(f'  {p}: {c}x')
        print()
        for _, row in errors.iterrows():
            ai_cols = [c for c in df_results.columns if c.startswith('ai_')]
            ps = ' '.join(f'{c[3:]}={row[c]}' for c in ai_cols if pd.notna(row.get(c)))
            print(f'  [{row["id"]}] {row["true_label"]:9s}->{row["moe_pred"]:9s} | {ps}')

### **15. Voting Method Analysis**

In [ ]:
if 'vote_info' in df_results.columns and 'true_label' in df_results.columns:
    methods = df_results['vote_info'].apply(lambda x: json.loads(x).get('method', '?'))
    print('Metodos:')
    for m, c in methods.value_counts().items():
        sub = df_results[methods == m]
        acc = (sub['moe_pred'] == sub['true_label']).mean()
        print(f'  {m:25s}: {c:3d} | acc={acc:.0%}')